In [34]:
import fitsio, re
from numpy.typing import NDArray, DTypeLike
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy import integrate
from copy import deepcopy
from astropy.io import fits

import euclidlib as el

from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations
from cloelib.observables.CometEFT_spectro import CometEFT_SpectroPower
from cloelike.EuclidLikelihood_3x2ptPlusGCspectro_ClsPlusPls import EuclidLikelihood_3x2ptPlusGCspectro_ClsPlusPls


In [3]:
# Execute this cell to download the data for testing the likelihood calculations
data_downloaded = True

if data_downloaded == False:
    
    import requests
    
    # URLs of the files to be downloaded
    urls = {
        'nz_example.fits': 'https://zenodo.org/records/15092862/files/nz_example.fits',
        'cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy': 'https://zenodo.org/records/15496892/files/cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy',
        'mixmat_identity_5000_binned.fits': 'https://zenodo.org/records/15496892/files/mixmat_identity_5000_binned.fits',
        'synth_cells_5000_binned.fits': 'https://zenodo.org/records/15496892/files/synth_cells_5000_binned.fits',
        'cov_Gauss_GCspectro_comet_EFT_z1.2_2500deg2.fits': 'https://zenodo.org/records/15495924/files/cov_Gauss_GCspectro_comet_EFT_z1.2_2500deg2.fits',
        'cov_Gauss_GCspectro_comet_EFT_z1.4_2500deg2.fits': 'https://zenodo.org/records/15495924/files/cov_Gauss_GCspectro_comet_EFT_z1.4_2500deg2.fits',
        'cov_Gauss_GCspectro_comet_EFT_z1.65_2500deg2.fits': 'https://zenodo.org/records/15495924/files/cov_Gauss_GCspectro_comet_EFT_z1.65_2500deg2.fits',
        'cov_Gauss_GCspectro_comet_EFT_z1._2500deg2.fits': 'https://zenodo.org/records/15495924/files/cov_Gauss_GCspectro_comet_EFT_z1._2500deg2.fits',
    }
    
    # Function to download a file
    def download_file(url, filename):
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f'{filename} downloaded successfully')
        else:
            print(f'Failed to download {filename}. Status code: {response.status_code}')
    
    # Download all files
    for filename, url in urls.items():
        download_file(url, filename)

In [19]:
# Get n(z)
z_nz, nz_heracles = el.photo.redshift_distributions('nz_example.fits')

# Normalize and resample n(z) for both position and shear
myz = np.linspace(1e-4, 3.0, 100)
def normalize_and_resample(nz_dict, z_grid, z_target):
    nz_array = np.vstack([nz / integrate.trapezoid(nz, z_grid) for nz in nz_dict.values()])
    return np.array([np.interp(z_target, z_grid, nz) for nz in nz_array])

my_dndz_pos_norm = normalize_and_resample(nz_heracles, z_nz, myz)
my_dndz_she_norm = normalize_and_resample(nz_heracles, z_nz, myz)

In [5]:
cells_data = el.photo.angular_power_spectra('synth_cells_5000_binned.fits')
mixmat = el.photo.mixing_matrices('mixmat_identity_5000_binned.fits')
full_cov=np.load('cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy')

In [6]:
data = {
    '3x2pt': {
        'cells':cells_data,
        'ells':cells_data['POS', 'POS', 1, 1].ell,
        'z_arr':myz,
        'dndz_pos':my_dndz_pos_norm,
        'dndz_she':my_dndz_she_norm,
        'cov':full_cov,
        'mixmat':mixmat}}

# Scale cuts have the same format as the data
scale_cut_dict = dict.fromkeys(cells_data.keys(), [10,1500])

for key in cells_data.keys():
    if key[:2]==('SHE','SHE'):
        scale_cut_dict[key]=[scale_cut_dict[key],[0,0]]
    
settings = {
    '3x2pt': {
        'n_ell_bins': 32,
        'scale_cuts': scale_cut_dict}}    

In [ ]:
redshifts = [1.0, 1.2, 1.4, 1.65]
labels = [str(z).strip('0') for z in redshifts]

data['GCspectro'] = {}
data['fiducial_cosmology'] = {
    'H0': 67.0,
    'Omega_cdm0': 0.27,
    'Omega_b0': 0.049,
    'Omega_k0': 0.0,
    'w0': -1.0,
    'wa': 0.0,
    'ns': 0.96,
    'mnu':0.00,
    'As': 2.1e-9,
    'gamma_MG': 0.545
}

fid_h = data['fiducial_cosmology']['H0'] / 100.0

def get_index(multipole, scale, scale_dict):
    offset = sum(len(scale_dict[ell]) for ell in multipoles if ell < multipole)
    return offset + np.where(scale_dict[multipole] == scale)[0][0]

k_fac = fid_h
pk_fac = 1.0 / fid_h**3
cov_fac = 1.0 / fid_h**6

nbar = np.array([2.042611E-03, 1.02876011E-03, 0.58531983E-03, 0.313402E-03])* fid_h**3

for ii,z in enumerate(labels):

    data['GCspectro'][z] = {}
    
    filename = f'cov_Gauss_GCspectro_comet_EFT_z{z}_2500deg2.fits'
    hdul = fits.open(filename)
    average = hdul['AVERAGE'].data
    data['GCspectro'][z]['k'] = average['SCALE_1DIM'] * k_fac
    data['GCspectro'][z]['pk0'] = average['AVERAGE0'] * pk_fac
    data['GCspectro'][z]['pk2'] = average['AVERAGE2'] * pk_fac
    data['GCspectro'][z]['pk4'] = average['AVERAGE4'] * pk_fac

    
    covariance = hdul['COVARIANCE'].data
    scale_i = covariance["SCALE_1DIM-I"]
    multipole_i = covariance["MULTIPOLE-I"]
    scale_j = covariance["SCALE_1DIM-J"]
    multipole_j = covariance["MULTIPOLE-J"]
    covariance = covariance["COVARIANCE"]

    multipoles = np.array([0, 2, 4])
    scale_dict = {ell: np.unique(scale_i[multipole_i == ell]) for ell in multipoles}
    matrix_size = sum(len(scale_dict[ell]) for ell in multipoles)
    cov_matrix = np.zeros((matrix_size, matrix_size))
    for s_i, m_i, s_j, m_j, cov in zip(scale_i, multipole_i, scale_j, multipole_j, covariance):
        i = get_index(m_i, s_i, scale_dict)
        j = get_index(m_j, s_j, scale_dict)
        cov_matrix[i, j] = cov
    data['GCspectro'][z]['cov'] = cov_matrix

    data['GCspectro'][z]['nbar'] = nbar[ii]

In [9]:
settings['GCspectro'] = {}

settings['GCspectro']['scale_cuts'] = {
    'bin1': {'ell0': [0.0, 0.20], 'ell2': [0.0, 0.15], 'ell4': [0.0, 0.15]},
    'bin2': {'ell0': [0.0, 0.25], 'ell2': [0.0, 0.20], 'ell4': [0.0, 0.20]},
    'bin3': {'ell0': [0.0, 0.25], 'ell2': [0.0, 0.20], 'ell4': [0.0, 0.20]},
    'bin4': {'ell0': [0.0, 0.30], 'ell2': [0.0, 0.25], 'ell4': [0.0, 0.25]}
}

for bin_key, bin_values in settings['GCspectro']['scale_cuts'].items():
    for ell_key in bin_values:
        values = bin_values[ell_key]
        bin_values[ell_key] = [v * fid_h for v in values]

In [18]:
import time

print("🚀 Initializing EuclidLikelihood_3x2ptPlusGCspectro_ClsPlusPls...")
start_time = time.perf_counter()

like_test = EuclidLikelihood_3x2ptPlusGCspectro_ClsPlusPls(
    data=data,
    settings=settings,
    Background=CAMBBackground,
    LinPerturbations=HMemuLinearPerturbations,
    NonLinPerturbations=HMemuNonLinearPerturbations,
    SpectroPower=CometEFT_SpectroPower
)

end_time = time.perf_counter()
elapsed = end_time - start_time
print(f"✅ Initialization complete! Took {elapsed:.3f} seconds 🎉")


🚀 Initializing EuclidLikelihood_3x2ptPlusGCspectro_ClsPlusPls...
✅ Initialization complete! Took 0.191 seconds 🎉


In [11]:
default_pars = {'H0':67,'Omega_cdm0':0.27,'Omega_b0':0.049,'ns':0.96,'As':2.1e-9,
                'w0':-1,'wa':0, 'Omega_k0':0,'mnu':0.0,'gamma_MG':0.545,
                'log10TAGN': 7.75,
                'AIA':0.16, 'EtaIA':1.66,
                'b1_photo_poly0': 1.33291, 'b1_photo_poly1': -0.72414,
                'b1_photo_poly2': 1.0183, 'b1_photo_poly3': -0.14913,
                'magnification_bias_1': 0.0, 'magnification_bias_2': 0.0,
                'magnification_bias_3': 0.0, 'magnification_bias_4': 0.0,
                'magnification_bias_5': 0.0, 'magnification_bias_6': 0.0,
                'dz_pos_1': 0.0, 'dz_pos_2': 0.0,
                'dz_pos_3': 0.0, 'dz_pos_4': 0.0,
                'dz_pos_5': 0.0, 'dz_pos_6': 0.0,
                'multiplicative_bias_1': 0.0, 'multiplicative_bias_2': 0.0,
                'multiplicative_bias_3': 0.0, 'multiplicative_bias_4': 0.0,
                'multiplicative_bias_5': 0.0, 'multiplicative_bias_6': 0.0,
                'dz_shear_1': 0.0, 'dz_shear_2': 0.0,
                'dz_shear_3': 0.0, 'dz_shear_4': 0.0,
                'dz_shear_5': 0.0, 'dz_shear_6': 0.0,
                'b1': np.array([1.412, 1.769, 2.039, 2.496]),
                'b2': np.array([0.695, 0.870, 1.162, 2.010]),
                'bG2': np.array([-0.156, -0.299, -0.400, -0.555]),
                'bGam3': np.array([0.323, 0.621, 0.827, 1.137]),
                'c0': np.array([30.948, 37.116, 36.738, 53.627]),
                'c2': np.array([46.233, 53.071, 48.626, 60.962]),
                'c4': np.array([10.057, 10.385, 8.643, 8.711]),
                'cnlo': np.array([0.0, 0.0, 0.0, 0.0]),
                'NP0': np.array([1.056, 1.152, 1.144, 1.309]),
                'NP20': np.array([0.0, 0.0, 0.0, 0.0]),
                'NP22': np.array([0.0, 0.0, 0.0, 0.0]),
                'fout': np.array([0.0, 0.0, 0.0, 0.0]),
                'sigmaz': np.array([0.0, 0.0, 0.0, 0.0])}

In [16]:
print("⏳ Starting log-likelihood evaluation...")
start_time = time.perf_counter()
result = like_test.loglike(default_pars)
end_time = time.perf_counter()

elapsed = end_time - start_time
print(f"✅ Log-likelihood result: {result:.6f}")
print(f"⏰ Elapsed time: {elapsed:.4f} seconds")



⏳ Starting log-likelihood evaluation...
✅ Log-likelihood result: -49.400474
⏰ Elapsed time: 0.5724 seconds
